# Episode 16: See What Your Agents Are Doing

When a single agent gives a wrong answer, you can read the prompt. When five agents collaborate on a wrong answer, good luck figuring out where things went sideways — unless you have observability.

In this episode, you'll add logging, tracing, and monitoring to your multi-agent systems so you always know what happened and why.

> Part of the **Production** section (Episodes 16–20).

## Why Observability Matters

Agent decisions are **opaque**. When a multi-agent system gives the wrong answer, you need to know which agent made the bad decision, what information it had, and why it routed the conversation that way. You also need to track how long each step took and how much it cost.

This isn't hypothetical. Amazon has built thousands of agents since 2025, and their teams consistently report that observability is non-negotiable at scale. Without it, debugging multi-agent systems is like debugging a distributed system with no logs.

### Three Levels of Observability

| Level | Question | Tools |
|-------|----------|-------|
| **Logging** | What happened? | Python `logging`, structured JSON logs |
| **Tracing** | How did it flow? | OpenTelemetry spans, distributed traces |
| **Monitoring** | Is it healthy? | Dashboards, alerts, cost tracking |

```
User Query
    │
    ├── [LOG]   "Received query: billing question"
    ├── [TRACE] Span: triage_agent (120ms)
    │       ├── [TRACE] Span: route_decision (45ms)
    │       └── [TRACE] Span: handoff_to_billing (5ms)
    ├── [TRACE] Span: billing_agent (850ms)
    │       ├── [TRACE] Span: tool_call: lookup_account (200ms)
    │       └── [TRACE] Span: generate_response (650ms)
    └── [MONITOR] Total: 970ms, 1,247 tokens, $0.002
```

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig

## Part 1: Python Logging

The simplest starting point is Python's built-in `logging` module. It won't show you distributed traces, but it gives you a timestamped record of what happened during a conversation.

We'll set up a logger, create a simple agent conversation (using the familiar `user.run(agent)` pattern), and inspect what gets logged.

In [ ]:
import logging

# Configure logging to show timestamps and source
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("ag2.workshop")

In [ ]:
# Create a simple agent and observe the log output
llm_config = LLMConfig({"model": "gpt-4o-mini"})

agent = ConversableAgent(
    name="helpful_assistant",
    system_message="You are a helpful assistant. Be concise.",
    llm_config=llm_config, human_input_mode="NEVER",
)

logger.info("Starting agent conversation")

user = ConversableAgent(name="user", human_input_mode="NEVER")

result = user.run(agent, message="What are the three primary colors?",
    max_turns=2,
)
result.process()

logger.info(f"Conversation completed. Cost: {result.cost}")
logger.info(f"Messages exchanged: {len(result.messages)}")

### Structured Logging with JSON

Plain text logs work for development. In production, structured JSON logs are what you actually want — they're machine-parseable, searchable, and every major platform (Elasticsearch, Datadog, CloudWatch) ingests them natively.

Here's a custom formatter that outputs each log record as a JSON object with optional agent metadata.

In [ ]:
import json
from datetime import datetime


class JSONFormatter(logging.Formatter):
    """Format log records as JSON for structured logging."""

    def format(self, record):
        log_data = {
            "timestamp": datetime.utcnow().isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        # Add extra fields if present
        if hasattr(record, "agent_name"):
            log_data["agent_name"] = record.agent_name
        if hasattr(record, "tokens"):
            log_data["tokens"] = record.tokens
        return json.dumps(log_data)


# Apply JSON formatter to a handler
json_handler = logging.StreamHandler()
json_handler.setFormatter(JSONFormatter())

json_logger = logging.getLogger("ag2.structured")
json_logger.addHandler(json_handler)
json_logger.setLevel(logging.INFO)

# Log with extra context
json_logger.info(
    "Agent conversation completed",
    extra={"agent_name": "helpful_assistant", "tokens": 150}
)

## Part 2: OpenTelemetry Tracing

Logging tells you *what happened*. **Tracing** tells you *how it flowed* — which agent called which, how long each step took, and where the bottlenecks hide.

[OpenTelemetry (OTel)](https://opentelemetry.io/) is the industry standard for distributed tracing, and AG2 has built-in support for it. Let's set up a basic tracer that outputs spans to the console.

In [ ]:
# pip install opentelemetry-api opentelemetry-sdk

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# Set up the tracer provider with console output
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

# Create a tracer for our application
tracer = trace.get_tracer("ag2.workshop.ep16")

### Creating Custom Spans

Spans are the building blocks of a trace. Each span represents one unit of work — an agent turn, a tool call, a handoff decision. Wrapping your agent operations in spans lets you track timing and attach metadata.

In [ ]:
import time

llm_config = LLMConfig({"model": "gpt-4o-mini"})

traced_agent = ConversableAgent(
    name="traced_assistant",
    system_message="You are a helpful assistant. Answer in one sentence.",
    llm_config=llm_config, human_input_mode="NEVER",
)

# Wrap the agent call in a trace span
with tracer.start_as_current_span("agent_conversation") as span:
    span.set_attribute("agent.name", "traced_assistant")
    span.set_attribute("agent.model", "gpt-4o-mini")

    start_time = time.time()

    user = ConversableAgent(name="user", human_input_mode="NEVER")

    result = user.run(traced_agent, message="What is the capital of France?",
        max_turns=2,
    )
    result.process()

    elapsed = time.time() - start_time
    span.set_attribute("agent.latency_ms", int(elapsed * 1000))
    span.set_attribute("agent.message_count", len(result.messages))

    print(f"\nCompleted in {elapsed:.2f}s")

### What to Trace

In a production multi-agent system, instrument these key points:

| What | Why | How |
|------|-----|-----|
| **Agent decisions** | Understand routing logic | Span per agent turn |
| **Tool calls** | Track external dependencies | Span per tool invocation |
| **Handoff reasons** | Debug routing issues | Attribute on handoff span |
| **Token usage** | Monitor costs | Attribute on agent span |
| **Latency per agent** | Find bottlenecks | Span duration |

AG2 has built-in OpenTelemetry support that instruments these automatically. See the AG2 blog posts on OTel tracing and event-driven logging for details.

## AgentOps Integration

If you want observability without building dashboards yourself, [AgentOps](https://www.agentops.ai/) provides a hosted platform purpose-built for agents — session replays, cost tracking, and error analysis out of the box.

```python
# pip install agentops
import agentops
agentops.init(api_key="your-key")

# Your AG2 code runs as normal — AgentOps captures everything automatically
result = agent.run(message="Hello!", max_turns=3)

agentops.end_session("Success")
```

AgentOps automatically captures every LLM call (prompt, response, tokens, cost), tool invocations and results, agent-to-agent handoffs, and full session timelines you can replay.

For more details, see the AG2 blog: [AgentOps Integration](https://ag2.ai/blog/2024-07-25-AgentOps).

## Try It Yourself

Go back to your Episode 10 customer service system and add tracing:

1. Set up an OpenTelemetry tracer
2. Wrap each agent interaction in a span
3. Track latency per agent
4. Identify which agent is the slowest

In [ ]:
# Your code here — add tracing to your customer service system

## What's Next

Now that you can see what your agents are doing, the next question is whether they're doing it safely. In **Episode 17**, you'll learn about **security for agent systems** — sandboxed code execution, input validation, and approval gates for critical actions.